# Imports

In [ ]:
import dataclasses
import json
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import benchmark_draws, paired_frame_bootstrap, score_draws, summarise
from uavloc.matching.correlator import PositionCorrelator
from uavloc.track import experiment_dir, set_seed

# Params

In [ ]:
EXP = "021-building-loss-from-scratch"
ARCHITECTURES = ["unet-resnet18", "unet-resnet34", "lraspp-mobilenetv3"]
LOSSES = ["current", "building", "structure"]
TRAIN_SEED = 42
STEPS = 2500
COMPARISONS = [("building", "current"), ("structure", "building"), ("structure", "current")]

WINDOW = "rect"                                      # correlator setup of 007.1 / 018.1
SEARCH_SCALE = 1.7
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"
MATCHING = "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)       # background, building, track, woody, field, water
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
HARM_MARGIN_M = 2.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"

GATE = "full"                                        # flight simulator's gated setup (outputs/check-gated, 007.8)
GATE_S_MAX_M = 3.0
GATE_C_MIN = 0.01
GATE_TEMPERATURE = 0.25

N_BOOTSTRAP = 10_000
CI_PCT = 95

# Data

In [ ]:
set_seed(SEED)
ARTIFACTS = experiment_dir(EXP, create=False) / "artifacts"
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "window": WINDOW, "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None,
    "gate": "none", "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
}
RUN_CONFIGS = {"ungated": RUN_CONFIG,
               "gated": {**RUN_CONFIG, "gate": GATE, "gate_s_max_m": GATE_S_MAX_M, "gate_c_min": GATE_C_MIN,
                         "temperature": GATE_TEMPERATURE}}
(ARTIFACTS / "correction-config.json").write_text(json.dumps(RUN_CONFIGS, indent=1))
decision_configs = {rule: correlator_config(c) for rule, c in RUN_CONFIGS.items()}
decision_configs

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
subsets = {"all": {f.name for f in frames}, **{p.split("-")[0]: {f.name for f in m} for p, m in places.items()}}
{place: len(members) for place, members in places.items()}

# Model

In [ ]:
checkpoints = {(arch, loss): ARTIFACTS / f"model-{arch}-{loss}-seed{TRAIN_SEED}-steps{STEPS}.pt"
               for arch in ARCHITECTURES for loss in LOSSES}
missing = [p.name for p in checkpoints.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"checkpoints not in {ARTIFACTS}: {missing}")
len(checkpoints)

In [ ]:
def correction_records(architecture: str, checkpoint: Path) -> pd.DataFrame:
    """Every draw with one segmenter on both sides, scored under each decision rule from the same surfaces."""
    model = load_segmenter(checkpoint, DEVICE, architecture)
    parts = []
    for place in PLACES:
        features = SegmentationFeatures(model, model, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)
        correlator = PositionCorrelator(features, decision_configs["ungated"])
        draws = [d for noise_m in NOISE_M
                 for d in benchmark_draws(correlator, places[place], noise_m, N_DIRECTIONS, SEED)]
        parts += [score_draws(draws, config).assign(place=place, rule=rule) for rule, config in decision_configs.items()]
    records = pd.concat(parts, ignore_index=True)
    records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
    return records

# Run

In [ ]:
records = {}
for (arch, loss), path in tqdm(checkpoints.items(), desc="models"):
    records[arch, loss] = correction_records(arch, path)
    records[arch, loss].to_csv(ARTIFACTS / f"correction-records-{arch}-{loss}-seed{TRAIN_SEED}.csv", index=False)

In [ ]:
def summary_row(records: pd.DataFrame) -> dict[str, float]:
    """Mean error per injected level and place, harm and decision shares at the primary level."""
    m = summarise(records, subsets, PRIMARY_NOISE_M)
    at = f"{PRIMARY_NOISE_M:g}"
    return {**{f"@{n:g} m": m[f"mean_position_error_m_at_{n:g}m"] for n in NOISE_M},
            **{f"{s} @{at}": m[f"mean_position_error_m_{s}"] for s in subsets if s != "all"},
            f"harm @{at}": m[f"harm_share_at_{at}m"],
            **{f"{d} @{at}": m[f"{d}_share_at_{at}m"] for d in ("accept", "across", "reject")}}

In [ ]:
summary = pd.DataFrame({(rule, arch, loss): summary_row(r[r["rule"] == rule])
                        for rule in RUN_CONFIGS for (arch, loss), r in records.items()}).T
summary.round(3)

In [ ]:
def rule_records(arch: str, loss: str, rule: str) -> pd.DataFrame:
    r = records[arch, loss]
    return r[r["rule"] == rule]

In [ ]:
paired = pd.DataFrame([{"rule": rule, "architecture": arch, "comparison": f"{new} - {ref}", "noise_m": n,
                        **paired_frame_bootstrap(rule_records(arch, new, rule), rule_records(arch, ref, rule), n,
                                                 N_BOOTSTRAP, CI_PCT, SEED)}
                       for rule in RUN_CONFIGS for arch in ARCHITECTURES for new, ref in COMPARISONS for n in NOISE_M])
paired[paired["noise_m"] == PRIMARY_NOISE_M].round(3)

In [ ]:
gate_effect = pd.DataFrame([{"architecture": arch, "loss": loss, "noise_m": n,
                             **paired_frame_bootstrap(rule_records(arch, loss, "gated"),
                                                      rule_records(arch, loss, "ungated"), n, N_BOOTSTRAP, CI_PCT, SEED)}
                            for arch in ARCHITECTURES for loss in LOSSES for n in NOISE_M])
gate_effect.round(3)

# Plots

In [ ]:
def plot_error_cdfs(rule: str, noise_m: float) -> plt.Figure:
    """Share of draws below each error at `noise_m` injected, one panel per architecture, one line per loss."""
    fig, axes = plt.subplots(1, len(ARCHITECTURES), figsize=(6 * len(ARCHITECTURES), 4), sharey=True, squeeze=False)
    for ax, arch in zip(axes[0], ARCHITECTURES):
        for loss in LOSSES:
            errors = np.sort(rule_records(arch, loss, rule).query("noise_m == @noise_m")["error_m"].to_numpy())
            ax.plot(errors, np.linspace(0, 1, len(errors)), label=loss)
        ax.axvline(noise_m, color="k", ls=":", lw=1)
        ax.set_title(f"{arch}, {rule}, {noise_m:g} m injected"); ax.set_xlabel("error after correction, m")
        ax.grid(alpha=.3); ax.legend()
    axes[0, 0].set_ylabel("share of draws")
    fig.tight_layout()
    return fig

In [ ]:
for rule in RUN_CONFIGS:
    plot_error_cdfs(rule, PRIMARY_NOISE_M)

In [ ]:
def plot_paired(rows: pd.DataFrame, label: pd.Series, title: str) -> plt.Figure:
    """Mean per-frame error difference with its bootstrap CI; below zero the first setup is better."""
    rows, label = rows.reset_index(drop=True), label.reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(rows) + 1))
    ax.errorbar(rows["diff_m"], rows.index, xerr=[rows["diff_m"] - rows["ci_lo"], rows["ci_hi"] - rows["diff_m"]],
                fmt="o", capsize=3)
    ax.set_yticks(rows.index, label)
    ax.axvline(0, color="k", lw=1)
    ax.invert_yaxis(); ax.grid(axis="x", alpha=.3)
    ax.set_xlabel(f"paired error difference, m ({CI_PCT}% frame bootstrap)"); ax.set_title(title)
    fig.tight_layout()
    return fig

In [ ]:
at_primary = paired[paired["noise_m"] == PRIMARY_NOISE_M]
plot_paired(at_primary, at_primary["rule"] + " " + at_primary["architecture"] + ": " + at_primary["comparison"],
            f"losses @ {PRIMARY_NOISE_M:g} m");

In [ ]:
gate_at_primary = gate_effect[gate_effect["noise_m"] == PRIMARY_NOISE_M]
plot_paired(gate_at_primary, gate_at_primary["architecture"] + " " + gate_at_primary["loss"],
            f"gated - ungated @ {PRIMARY_NOISE_M:g} m");